**Goal:** Visually explore patterns in delivery performance, 
identify SLA breach drivers, and produce various charts 
that tell a complete business story.

**Input:** `../data/processed/orders_clean.csv`  
**Output:** Charts saved to `../dashboard/screenshots/`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')

# global chart styling — consistent look across all charts
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams.update({
    'figure.dpi': 150,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.titleweight': 'bold',
    'axes.labelsize': 11,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10
})

OUTPUT_DIR = '../dashboard/screenshots/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete. Charts will save to:", OUTPUT_DIR)

In [ ]:
df = pd.read_csv('../data/processed/orders_clean.csv')

# restore correct dtypes after CSV round-trip
df['sla_breached'] = df['sla_breached'].astype(bool)
df['is_weekend']   = df['is_weekend'].astype(bool)
df['order_date']   = pd.to_datetime(df['order_date'], errors='coerce')

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nSLA breach rate: {df['sla_breached'].mean()*100:.2f}%")
print(f"Avg delivery time: {df['time_taken_min'].mean():.1f} min")
print(f"Median delivery time: {df['time_taken_min'].median():.0f} min")

### Chart 1: Delivery Time Distribution

**What we are looking for:**
- Is the distribution normal, right-skewed, or bimodal?
- Where does the bulk of orders fall relative to the 30-min SLA?
- What % of orders are clearly fast vs clearly late?

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

ax.hist(
    df['time_taken_min'],
    bins=40,
    color='#4A90D9',
    edgecolor='white',
    linewidth=0.5,
    alpha=0.85
)

# SLA threshold line
ax.axvline(x=30, color='#E74C3C', linewidth=2.5,
           linestyle='--', label='30-min SLA threshold')

# Median line
median_val = df['time_taken_min'].median()
ax.axvline(x=median_val, color='#27AE60', linewidth=2,
           linestyle='-', label=f'Median: {median_val:.0f} min')

# Annotations
breach_pct = df['sla_breached'].mean() * 100
ax.text(35, ax.get_ylim()[1]*0.85,
        f'{breach_pct:.1f}% orders\nbreach SLA',
        color='#E74C3C', fontsize=10, fontweight='bold')

ax.set_title('Distribution of Delivery Time (minutes)')
ax.set_xlabel('Delivery Time (minutes)')
ax.set_ylabel('Number of Orders')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart01_delivery_time_distribution.png',
            bbox_inches='tight')
plt.show()
print("Chart 1 saved.")

**Business Interpretation - Chart 1:**
Delivery times are spread evenly between 20–50 minutes, showing no strong focus around the 30-minute SLA target.
Although the median delivery time is 26 minutes (below the SLA), a large percentage of orders still exceed 30 minutes. This suggests the SLA is being followed loosely rather than strictly.


### Chart 2: SLA Breach Rate by City Tier

In [ ]:
city_stats = (
    df.groupby('city')
    .agg(
        total_orders   = ('order_id', 'count'),
        breach_pct     = ('sla_breached', lambda x: round(x.mean()*100, 2)),
        avg_time       = ('time_taken_min', 'mean')
    )
    .reset_index()
    .sort_values('breach_pct', ascending=False)
)

fig, ax = plt.subplots(figsize=(9, 5))

bars = ax.bar(
    city_stats['city'],
    city_stats['breach_pct'],
    color=['#E74C3C' if v > 50 else '#4A90D9' for v in city_stats['breach_pct']],
    edgecolor='white', width=0.5
)

# add value labels on bars
for bar, val in zip(bars, city_stats['breach_pct']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{val:.1f}%',
            ha='center', va='bottom',
            fontweight='bold', fontsize=11)

ax.axhline(y=df['sla_breached'].mean()*100,
           color='grey', linestyle='--', linewidth=1.5,
           label=f'Overall avg: {df["sla_breached"].mean()*100:.1f}%')

ax.set_title('SLA Breach Rate by City Tier')
ax.set_xlabel('City Tier')
ax.set_ylabel('SLA Breach Rate (%)')
ax.set_ylim(0, 100)
ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart02_sla_breach_by_city.png', bbox_inches='tight')
plt.show()

print("\nCity stats:")
print(city_stats.to_string(index=False))

**Business Interpretation - Chart 2:** Metropolitan cities show the highest SLA breach rate at **X%**, followed by Urban cities at **Y%**. This indicates that higher order volume and traffic congestion in metro areas are putting more pressure on delivery operations compared to smaller city tiers.


### Chart 3: SLA Breach Rate by Weather Condition

In [ ]:
weather_stats = (
    df.groupby('weather')
    .agg(
        total_orders = ('order_id', 'count'),
        breach_pct   = ('sla_breached', lambda x: round(x.mean()*100, 2)),
        avg_time     = ('time_taken_min', 'mean')
    )
    .reset_index()
    .sort_values('breach_pct', ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# left: breach rate by weather
colors = sns.color_palette('RdYlGn_r', len(weather_stats))
axes[0].barh(weather_stats['weather'], weather_stats['breach_pct'],
             color=colors, edgecolor='white')
for i, (val, total) in enumerate(zip(weather_stats['breach_pct'],
                                      weather_stats['total_orders'])):
    axes[0].text(val + 0.3, i, f'{val:.1f}%  (n={total:,})',
                 va='center', fontsize=9)
axes[0].set_title('SLA Breach Rate by Weather')
axes[0].set_xlabel('Breach Rate (%)')
axes[0].set_xlim(0, 105)

# right: avg delivery time by weather
axes[1].barh(weather_stats['weather'],
             weather_stats['avg_time'].round(1),
             color='#4A90D9', edgecolor='white')
axes[1].axvline(x=30, color='red', linestyle='--',
                linewidth=1.5, label='30-min SLA')
for i, val in enumerate(weather_stats['avg_time']):
    axes[1].text(val + 0.2, i, f'{val:.1f} min',
                 va='center', fontsize=9)
axes[1].set_title('Avg Delivery Time by Weather')
axes[1].set_xlabel('Avg Delivery Time (min)')
axes[1].legend()

plt.suptitle('Weather Conditions Impact on Delivery Performance',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart03_weather_impact.png', bbox_inches='tight')
plt.show()

**Business Interpretation - Chart 3:** Stormy and Sandstorm conditions show much higher SLA breach rates compared to Sunny weather, indicating that bad weather is a major factor behind delayed deliveries. To reduce SLA failures, the operations team should increase delivery partner availability and improve route planning during forecasted adverse weather conditions.


### Chart 4: SLA Breach Rate by Traffic Density

In [ ]:
# define correct traffic order
traffic_order = ['Low', 'Medium', 'High', 'Jam']

traffic_stats = (
    df[df['traffic_density'].isin(traffic_order)]
    .groupby('traffic_density')
    .agg(
        total_orders = ('order_id', 'count'),
        breach_pct   = ('sla_breached', lambda x: round(x.mean()*100, 2)),
        avg_time     = ('time_taken_min', 'mean')
    )
    .reindex(traffic_order)
    .reset_index()
)

fig, ax = plt.subplots(figsize=(9, 5))

bar_colors = ['#27AE60', '#F1C40F', '#E67E22', '#E74C3C']
bars = ax.bar(traffic_stats['traffic_density'],
              traffic_stats['breach_pct'],
              color=bar_colors, edgecolor='white', width=0.5)

for bar, val, n in zip(bars, traffic_stats['breach_pct'],
                        traffic_stats['total_orders']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.8,
            f'{val:.1f}%\n(n={n:,})',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('SLA Breach Rate by Traffic Density\n'
             '(Green = Low risk → Red = High risk)')
ax.set_xlabel('Traffic Density')
ax.set_ylabel('SLA Breach Rate (%)')
ax.set_ylim(0, 110)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart04_traffic_impact.png', bbox_inches='tight')
plt.show()

**Business Interpretation - Chart 4:** SLA breach rates rise steadily from Low to Jam traffic conditions, with Jam traffic showing the highest breach rate at nearly **X%**, compared to **Y%** under Low traffic. This highlights traffic congestion as a key operational factor affecting delivery performance. Using real-time traffic insights for better partner allocation and smarter order routing can help reduce delays and improve SLA adherence.


### Chart 5: Peak Hour Heatmap - Hour × Day of Week

In [ ]:
# filter to rows with valid date and hour
heatmap_df = df[
    (df['order_day_of_week'] != 'Unknown') &
    (df['order_hour'].notna())
].copy()

# build pivot table: rows = day, columns = hour, values = breach %
day_order = ['Monday','Tuesday','Wednesday','Thursday',
             'Friday','Saturday','Sunday']

pivot = (
    heatmap_df
    .groupby(['order_day_of_week', 'order_hour'])['sla_breached']
    .mean()
    .mul(100)
    .round(1)
    .reset_index()
    .pivot(index='order_day_of_week', columns='order_hour',
           values='sla_breached')
    .reindex(day_order)
)

fig, ax = plt.subplots(figsize=(16, 6))

sns.heatmap(
    pivot,
    annot=True,
    fmt='.0f',
    cmap='RdYlGn_r',
    linewidths=0.4,
    linecolor='white',
    cbar_kws={'label': 'SLA Breach Rate (%)'},
    ax=ax,
    vmin=0, vmax=100
)

ax.set_title('SLA Breach Rate (%) by Hour of Day and Day of Week\n'
             'Red = High breach | Green = Low breach',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Hour of Day (0 = midnight, 12 = noon)')
ax.set_ylabel('Day of Week')

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart05_heatmap_hour_day.png', bbox_inches='tight')
plt.show()
print("Heatmap saved.")

**Business Interpretation — Chart 5:** The heatmap shows that SLA breach rates are highest during evening peak hours, especially between **8–10 PM**, with **Friday evenings** recording the maximum concentration of delays. In contrast, early morning hours (**6–9 AM**) consistently have the lowest breach rates across all weekdays. This indicates that order surges during dinner hours are overwhelming delivery capacity, making peak-time staffing and smarter partner allocation critical for maintaining SLA performance.


### Chart 6: Delivery Speed Category Distribution

In [ ]:
speed_order = ['Fast', 'On-time', 'Late', 'Very Late']
speed_colors = ['#27AE60', '#4A90D9', '#E67E22', '#E74C3C']

speed_counts = df['delivery_speed'].value_counts().reindex(speed_order)
speed_pcts   = (speed_counts / len(df) * 100).round(1)

fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(speed_order, speed_pcts.values,
              color=speed_colors, edgecolor='white', width=0.55)

for bar, val, count in zip(bars, speed_pcts.values, speed_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.5,
            f'{val:.1f}%\n({count:,} orders)',
            ha='center', va='bottom',
            fontsize=10, fontweight='bold')

ax.set_title('Order Distribution by Delivery Speed Category\n'
             'Fast (<20min) | On-time (20–30min) | '
             'Late (30–45min) | Very Late (>45min)')
ax.set_xlabel('Delivery Speed Category')
ax.set_ylabel('Percentage of Total Orders (%)')
ax.set_ylim(0, max(speed_pcts.values) * 1.2)

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart06_speed_category.png', bbox_inches='tight')
plt.show()

**Business Interpretation - Chart 6:** Only **X%** of orders are delivered within 20 minutes, while the largest share — **Y%** — falls into the Late category (30–45 minutes). This suggests that most delays are moderate rather than severe. From an operational perspective, this is a positive sign because even a small improvement of 5–10 minutes in delivery efficiency could significantly increase overall SLA compliance.


### Chart 12: SLA Breach Rate by Hour (Line Chart)

In [ ]:
hourly = (
    df[df['order_hour'].notna()]
    .groupby('order_hour')
    .agg(
        total_orders = ('order_id', 'count'),
        breach_pct   = ('sla_breached', lambda x: x.mean()*100)
    )
    .reset_index()
)

fig, ax1 = plt.subplots(figsize=(13, 5))

# line: breach rate
color1 = '#E74C3C'
ax1.plot(hourly['order_hour'], hourly['breach_pct'],
         color=color1, linewidth=2.5, marker='o',
         markersize=5, label='SLA Breach Rate (%)')
ax1.fill_between(hourly['order_hour'], hourly['breach_pct'],
                 alpha=0.12, color=color1)
ax1.set_xlabel('Hour of Day')
ax1.set_ylabel('SLA Breach Rate (%)', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.set_ylim(0, 100)

# bar: order volume
ax2 = ax1.twinx()
ax2.bar(hourly['order_hour'], hourly['total_orders'],
        alpha=0.2, color='#4A90D9', label='Order Volume')
ax2.set_ylabel('Number of Orders', color='#4A90D9')
ax2.tick_params(axis='y', labelcolor='#4A90D9')

# combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2,
           loc='upper left', fontsize=9)

ax1.set_title('SLA Breach Rate and Order Volume by Hour of Day\n'
              '(Dual axis — Red line = Breach %, Blue bars = Order count)')
ax1.set_xticks(range(0, 24))

plt.tight_layout()
plt.savefig(OUTPUT_DIR + 'chart07_hourly_breach_trend.png', bbox_inches='tight')
plt.show()

**Business Interpretation - Chart 7:** SLA breach rates increase significantly during evening hours (**7–10 PM**), which also coincide with the highest order volumes. This clearly indicates a supply-demand imbalance, where delivery capacity is unable to keep up with peak-time demand. In comparison, morning hours (**9–11 AM**) handle relatively high order volumes with lower breach rates, suggesting that operations during this period are more efficiently staffed and managed.
